In [ ]:
import numpy as np
import matplotlib.pyplot as plt  
import pandas as pd
from IPython.display import display

from sklearn.cluster import DBSCAN
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances
from scipy.sparse.linalg import eigsh

from brainpred import paths, splits
from brainpred.utils import display as vis
from brainpred.utils import intensity, metrics, ml as uml
from brainpred.features.inter_patient import tissue_based as inter_patient_tissue_based

# The prototype similarity tables (test_code/dfield_approach_elastix/approach_images_similar/similarities)
# were replaced by the inter-patient tissue tables (96 tissues, background excluded, plus absdif_* columns):
# 'dice_mean' is close to, but not identical with, the prototype one.
def SimilarityTablesReader(ids):
    return inter_patient_tissue_based.FeaturesLoader(96, ids)

## Read similarity table

In [ ]:
matching_ids = splits.get_matching_ids()
sim_reader = SimilarityTablesReader(matching_ids)
sim_np = sim_reader.similarity_np

# np.fill_diagonal(sim_np, 1)
# sim_table_np = 1 / sim_table_np
# sim_table_np = 1 - sim_table_np
print(sim_np.shape)

print(f"min: {np.min(sim_np)}\t max: {np.max(sim_np)}\t mean: {np.mean(sim_np)}")

print(sim_np[:5, :5, 2])

## Find best clustering

In [ ]:


def spectral_clustering_with_multi_metrics(similarity_matrix, num_clusters):
    # Obtener la matriz de afinidad sumando las matrices de similitud para cada métrica
    affinity_matrix = np.sum(similarity_matrix, axis=2)
    
    # Calcular la matriz laplaciana normalizada
    degree_matrix = np.diag(np.sum(affinity_matrix, axis=1))
    laplacian_matrix = degree_matrix - affinity_matrix
    normalized_laplacian = np.dot(np.linalg.inv(degree_matrix), laplacian_matrix)
    
    # Calcular los autovectores correspondientes a los autovalores más grandes
    num_eigenvectors = num_clusters + 1  # Se agrega uno para excluir el autovector constante
    eigenvalues, eigenvectors = eigsh(normalized_laplacian, k=num_eigenvectors, which='SM')
    
    # Clustering de los autovectores utilizando KMeans
    kmeans = KMeans(n_clusters=num_clusters)
    kmeans.fit(eigenvectors[:, 1:])  # Se excluye el autovector constante
    
    # Devolver las etiquetas de los clusters para cada elemento
    return kmeans.labels_

labels = spectral_clustering_with_multi_metrics(sim_np[:,:,2:5], 2)
print(labels)